# HyFIN-Net — Hyper-Frequency Inception Network for MERC

Single-notebook implementation of the architecture in `architecture.md`. Combines:
- **Inception Graph Module (IGM)** with multi-window k-GNN branches (ConxGNN)
- **Hypergraph Module (HM)** with edge-dependent node weights (M3Net + ConxGNN)
- **Multi-Frequency Module** with low-/high-pass self-gating (M3Net)
- **Implicit-edge detector** (HRG-SSA, lightweight FFN + masked softmax)
- **Cross-modal attention fusion** (text-anchored, ConxGNN)
- Losses: **CBCE** (class-balanced CE) + **CBFC** (supervised focal-contrastive over the penultimate reps, ConxGNN Eq.18) + DualCL (placeholder)

Tested input pickles (Kaggle):
```
/kaggle/input/datasets/gilbertstrange/graphsmile-preprocessed/GraphSmile_PreProcessed/iemocap_multi_features.pkl
/kaggle/input/datasets/gilbertstrange/graphsmile-preprocessed/GraphSmile_PreProcessed/meld_multi_features.pkl
```

## 0. Environment

In [ ]:
import subprocess, sys, importlib
def _try(mod):
    try:
        importlib.import_module(mod); return True
    except Exception:
        return False
# Kaggle GPU image already ships torch + torch_geometric, but torch_scatter/torch_sparse may be missing.
if not _try('torch_scatter') or not _try('torch_sparse'):
    import torch as _t
    tv = _t.__version__.split('+')[0]
    cu = _t.version.cuda.replace('.', '') if _t.version.cuda else 'cpu'
    url = f'https://data.pyg.org/whl/torch-{tv}+cu{cu}.html'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-index',
                    'torch_scatter', 'torch_sparse', '-f', url], check=False)
    if not _try('torch_geometric'):
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch_geometric'], check=False)
import torch, torch_geometric
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| pyg', torch_geometric.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
import os
IS_KAGGLE = os.path.exists('/kaggle/working')
print('Platform: Kaggle' if IS_KAGGLE else 'Platform: local')

In [ ]:
import os, pickle, math, random, time, copy, json
from pathlib import Path
from itertools import permutations
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch_geometric.nn import GraphConv, TransformerConv
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax as pyg_softmax, degree
from torch_scatter import scatter_add
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix

SEED = 2024
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device)


## 1. Config

In [ ]:
# ── Local paths ── edit DATA_ROOT or set env vars when running outside Kaggle ─
# Override via: export GRAPHSMILE_DATA=/path/to/GraphSmile_PreProcessed
#               export HYFIN_SAVE_DIR=/path/to/outputs
_LOCAL_DATA_ROOT = os.environ.get(
    'GRAPHSMILE_DATA',
    str(Path('/mnt/Work/ML/Thesis/WACV/data/GraphSmile_PreProcessed')))
_LOCAL_SAVE_DIR  = os.environ.get('HYFIN_SAVE_DIR', './outputs')
# ─────────────────────────────────────────────────────────────────────────────

class Cfg:
    # ---- choose dataset: 'meld' or 'iemocap'
    dataset      = 'iemocap'
    data_root    = ('/kaggle/input/datasets/gilbertstrange/graphsmile-preprocessed/GraphSmile_PreProcessed'
                    if IS_KAGGLE else _LOCAL_DATA_ROOT)
    save_dir     = '/kaggle/working' if IS_KAGGLE else _LOCAL_SAVE_DIR
    @property
    def meld_path(self):    return f'{self.data_root}/meld_multi_features.pkl'
    @property
    def iemocap_path(self): return f'{self.data_root}/iemocap_multi_features.pkl'
    # ---- training
    batch_size_d = {'iemocap': 16, 'meld': 32}
    epochs       = 60
    lr           = 4e-4
    weight_decay = 1e-4
    grad_clip    = 1.0
    warmup_epochs = 1
    label_smoothing = 0.1
    # ---- model (compact: hidden 256 ~ 4-5M params)
    hidden       = 256
    n_speakers   = {'iemocap': 2, 'meld': 9}
    n_classes    = {'iemocap': 6, 'meld': 7}
    # IGM
    igm_branches = {
        'iemocap': [(5, 3), (3, 2)],
        'meld':    [(7, 4), (5, 3)],
    }
    igm_layers   = 2
    igm_heads    = 4          # TransformerConv heads (d / heads must be int)
    # HM — match M3Net spec: 2 for IEMOCAP, 4 for MELD
    hm_layers    = {'iemocap': 2, 'meld': 4}
    # Multi-Frequency — match M3Net spec: 4 for IEMOCAP, 3 for MELD
    mf_layers    = {'iemocap': 4, 'meld': 3}
    # Cross-modal attention
    ca_heads     = 4
    # Dropout (higher — model overfits)
    dropout      = {'iemocap': 0.5, 'meld': 0.5}
    # Losses
    beta_cb      = 0.999
    cbfc_mu      = 0.1
    cbfc_gamma   = 2.0
    cbfc_temp    = 0.5
    dualcl_lam   = 0.0
    dualcl_tau   = 0.5
    dualcl_drop  = 0.1
    # Caps for HyperGraphModule learnable weight tables
    max_hm_nodes = 50_000
    max_hm_edges = 50_000
    val_frac     = 0.1
    log_every    = 100
    @property
    def batch_size(self): return self.batch_size_d[self.dataset]

cfg = Cfg()
os.makedirs(cfg.save_dir, exist_ok=True)
print(f'data_root: {cfg.data_root}')
print(f'save_dir : {cfg.save_dir}')


## 2. Dataset — robust loader for GraphSmile preprocessed pickles

GraphSmile's preprocessed pickle follows the standard MMGCN/M3Net layout:
```
(videoIDs, videoSpeakers, videoLabels, videoText, videoAudio, videoVisual,
 videoSentence, trainVid, testVid[, _])
```
Each `video*` is `dict[vid] -> list/ndarray` indexed by utterance position.

In [ ]:
def _load_pickle(path):
    with open(path, 'rb') as f:
        try:
            return pickle.load(f, encoding='latin1')
        except TypeError:
            f.seek(0); return pickle.load(f)

def parse_graphsmile_pickle(path, dataset):
    """Supports all three GraphSmile preprocessed layouts:
      9  : IEMOCAPDataset_BERT4 (single videoText)
      12 : IEMOCAPDataset_BERT  (videoText0..3)
      13/14 : MELDDataset_BERT  (videoSentiments + videoText0..3 + trailing _)
    For multi-layer text we average the 4 RoBERTa layers (M3Net `(r1+r2+r3+r4)/4`).
    """
    obj = _load_pickle(path)
    assert isinstance(obj, (tuple, list)), f'unexpected pickle type {type(obj)}'
    n = len(obj)
    if n == 9:
        (videoIDs, videoSpeakers, videoLabels,
         videoText, videoAudio, videoVisual, videoSentence,
         trainVid, testVid) = obj
        text_dict = videoText
    elif n in (12,):
        (videoIDs, videoSpeakers, videoLabels,
         vT0, vT1, vT2, vT3,
         videoAudio, videoVisual, videoSentence,
         trainVid, testVid) = obj
        text_dict = {vid: (np.asarray(vT0[vid], dtype=np.float32)
                          + np.asarray(vT1[vid], dtype=np.float32)
                          + np.asarray(vT2[vid], dtype=np.float32)
                          + np.asarray(vT3[vid], dtype=np.float32)) / 4.0
                     for vid in vT0.keys()}
    elif n in (13, 14):
        (videoIDs, videoSpeakers, videoLabels, videoSentiments,
         vT0, vT1, vT2, vT3,
         videoAudio, videoVisual, videoSentence,
         trainVid, testVid, *rest) = obj
        text_dict = {vid: (np.asarray(vT0[vid], dtype=np.float32)
                          + np.asarray(vT1[vid], dtype=np.float32)
                          + np.asarray(vT2[vid], dtype=np.float32)
                          + np.asarray(vT3[vid], dtype=np.float32)) / 4.0
                     for vid in vT0.keys()}
    else:
        raise ValueError(f'unsupported pickle layout: {n} fields')

    sample_vid = next(iter(text_dict))
    Dt = int(np.asarray(text_dict[sample_vid]).shape[-1])
    Da = int(np.asarray(videoAudio[sample_vid]).shape[-1])
    Dv = int(np.asarray(videoVisual[sample_vid]).shape[-1])
    print(f'  layout={n} fields  dims t/a/v = {Dt}/{Da}/{Dv}  train={len(trainVid)}, test={len(testVid)}')
    return dict(text=text_dict, audio=videoAudio, visual=videoVisual,
                labels=videoLabels, speakers=videoSpeakers, sentences=videoSentence,
                trainVid=list(trainVid), testVid=list(testVid))

def _speaker_to_idx(spk, dataset):
    if dataset == 'iemocap':
        out = []
        for s in spk:
            if isinstance(s, (str, bytes)):
                ss = s.decode() if isinstance(s, bytes) else s
                out.append({'M': 0, 'F': 1}.get(ss, 0))
            else:
                out.append(int(s))
        return out
    arr = np.asarray(spk)
    if arr.ndim == 2:                # MELD: one-hot rows
        return arr.argmax(-1).tolist()
    return arr.astype(int).tolist()

class MERCDataset(Dataset):
    def __init__(self, raw, vids, dataset):
        self.raw = raw; self.vids = vids; self.dataset = dataset
    def __len__(self): return len(self.vids)
    def __getitem__(self, idx):
        vid = self.vids[idx]
        t = np.asarray(self.raw['text'][vid],   dtype=np.float32)
        a = np.asarray(self.raw['audio'][vid],  dtype=np.float32)
        v = np.asarray(self.raw['visual'][vid], dtype=np.float32)
        y = np.asarray(self.raw['labels'][vid], dtype=np.int64)
        spk = _speaker_to_idx(self.raw['speakers'][vid], self.dataset)
        assert t.ndim == 2 and a.ndim == 2 and v.ndim == 2, (
            f'vid={vid} shapes t{t.shape} a{a.shape} v{v.shape}')
        return {
            'text': torch.from_numpy(t), 'audio': torch.from_numpy(a),
            'visual': torch.from_numpy(v), 'label': torch.from_numpy(y),
            'speaker': torch.tensor(spk, dtype=torch.long),
            'length': int(t.shape[0]),
        }

def pad_collate(batch):
    B = len(batch)
    L = max(s['length'] for s in batch)
    Dt = batch[0]['text'].shape[-1]
    Da = batch[0]['audio'].shape[-1]
    Dv = batch[0]['visual'].shape[-1]
    text = torch.zeros(B, L, Dt)
    audio = torch.zeros(B, L, Da)
    visual = torch.zeros(B, L, Dv)
    spk = torch.zeros(B, L, dtype=torch.long)
    lens = torch.zeros(B, dtype=torch.long)
    labels = []
    for i, s in enumerate(batch):
        n = s['length']
        text[i, :n]   = s['text']
        audio[i, :n]  = s['audio']
        visual[i, :n] = s['visual']
        spk[i, :n]    = s['speaker']
        lens[i]       = n
        labels.append(s['label'])
    return {'text': text, 'audio': audio, 'visual': visual,
            'speaker': spk, 'lengths': lens, 'labels': torch.cat(labels)}

# --- load and probe ---
path = cfg.meld_path if cfg.dataset == 'meld' else cfg.iemocap_path
print(f'loading {path}')
raw = parse_graphsmile_pickle(path, dataset=cfg.dataset)
print('train vids:', len(raw['trainVid']), 'test vids:', len(raw['testVid']))

probe_dims = []
for vid in list(raw['trainVid'])[:5]:
    t = np.asarray(raw['text'][vid]); a = np.asarray(raw['audio'][vid]); v = np.asarray(raw['visual'][vid])
    assert t.ndim == a.ndim == v.ndim == 2, f'vid={vid} non-2D features: t{t.shape} a{a.shape} v{v.shape}'
    probe_dims.append((t.shape[-1], a.shape[-1], v.shape[-1]))
assert len(set(probe_dims)) == 1, f'inconsistent feature dims across dialogues: {set(probe_dims)}'
D_T, D_A, D_V = probe_dims[0]
print(f'consistent feature dims t/a/v = {D_T}/{D_A}/{D_V}')


In [ ]:
# train/val/test split (no official val on IEMOCAP/MELD GraphSmile → carve from train)
train_all = list(raw['trainVid'])
rng = random.Random(SEED); rng.shuffle(train_all)
n_val = max(1, int(len(train_all) * cfg.val_frac))
val_vids = train_all[:n_val]
train_vids = train_all[n_val:]
test_vids = list(raw['testVid'])

train_set = MERCDataset(raw, train_vids, cfg.dataset)
val_set   = MERCDataset(raw, val_vids,   cfg.dataset)
test_set  = MERCDataset(raw, test_vids,  cfg.dataset)

train_loader = DataLoader(train_set, batch_size=cfg.batch_size, shuffle=True,
                          collate_fn=pad_collate, num_workers=(2 if IS_KAGGLE else min(4, os.cpu_count() or 1)), pin_memory=IS_KAGGLE)
val_loader   = DataLoader(val_set,   batch_size=cfg.batch_size, shuffle=False,
                          collate_fn=pad_collate, num_workers=(2 if IS_KAGGLE else min(4, os.cpu_count() or 1)), pin_memory=IS_KAGGLE)
test_loader  = DataLoader(test_set,  batch_size=cfg.batch_size, shuffle=False,
                          collate_fn=pad_collate, num_workers=(2 if IS_KAGGLE else min(4, os.cpu_count() or 1)), pin_memory=IS_KAGGLE)
print(f'split: train={len(train_set)}  val={len(val_set)}  test={len(test_set)}')

# class counts on train for class-balanced losses
class_counts = np.zeros(cfg.n_classes[cfg.dataset], dtype=np.int64)
for vid in train_vids:
    for y in raw['labels'][vid]:
        class_counts[int(y)] += 1
print('train class counts:', class_counts.tolist())

## 3. Unimodal Encoder (Block A)
1-layer Transformer for text; FC+ReLU for audio/visual; additive speaker embedding.

In [ ]:
class InputLN(nn.Module):
    """Masked per-dialogue LayerNorm over each dialogue's TRUE (unpadded) (n, D)
    block, without learnable affine. Matches M3Net's `LN2`, which runs on the
    real sequence because M3Net processes one (unpadded) dialogue at a time.

    BUGFIX: the previous version applied `nn.LayerNorm((L, D))` to the padded
    [B, L, D] tensor *before* masking, so each dialogue's mean/variance were
    computed over L rows including (L - n) zero-padding rows. Since L is the
    batch's max length, the same dialogue normalized DIFFERENTLY depending on
    which other dialogues shared its batch — non-deterministic across batchings
    and a contamination of the M3Net-style LN2. Here we compute statistics over
    only the n real utterances (n*D real elements) per dialogue, so the result
    is independent of batch composition and padding.
    """
    def __init__(self, eps=1e-5):
        super().__init__()
        self.eps = eps

    def forward(self, x, lengths):  # x: [B, L, D], lengths: [B]
        B, L, D = x.shape
        if L == 0:
            return x
        # valid (real-utterance) mask -> [B, L, 1]
        valid = (torch.arange(L, device=x.device)[None, :] < lengths[:, None]).unsqueeze(-1)
        vf = valid.to(x.dtype)
        # number of real scalar elements per dialogue = n * D  (>=1 to avoid /0)
        cnt = (vf.sum(dim=(1, 2), keepdim=True) * D).clamp(min=1.0)  # [B,1,1]
        mean = (x * vf).sum(dim=(1, 2), keepdim=True) / cnt
        var = (((x - mean) ** 2) * vf).sum(dim=(1, 2), keepdim=True) / cnt
        out = (x - mean) / torch.sqrt(var + self.eps)
        return out * vf  # keep padding rows at zero (they are discarded later)

class PositionalEncoding(nn.Module):
    def __init__(self, d, max_len=200):
        super().__init__()
        pe = torch.zeros(max_len, d)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))  # [1, max_len, d]
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class UnimodalEncoder(nn.Module):
    def __init__(self, d_t, d_a, d_v, d_h, n_speakers, dropout=0.5):
        super().__init__()
        self.in_ln_t = InputLN()
        self.in_ln_a = InputLN()
        self.in_ln_v = InputLN()
        self.t_proj = nn.Linear(d_t, d_h)
        self.pe     = PositionalEncoding(d_h)
        self.t_enc  = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=d_h, nhead=4, dim_feedforward=d_h,
                                       dropout=dropout, activation='gelu', batch_first=True),
            num_layers=1)
        self.a_proj = nn.Sequential(nn.Linear(d_a, d_h), nn.ReLU(), nn.Dropout(dropout))
        self.v_proj = nn.Sequential(nn.Linear(d_v, d_h), nn.ReLU(), nn.Dropout(dropout))
        self.spk    = nn.Embedding(n_speakers, d_h)

    def forward(self, text, audio, visual, spk, lengths):
        B, L, _ = text.shape
        mask = torch.arange(L, device=text.device)[None] >= lengths[:, None]  # True = pad
        # M3Net-style per-dialogue LN on raw features, computed over each
        # dialogue's real utterances only (masked; batch-independent).
        text   = self.in_ln_t(text,   lengths)
        audio  = self.in_ln_a(audio,  lengths)
        visual = self.in_ln_v(visual, lengths)
        ht = self.t_enc(self.pe(self.t_proj(text)), src_key_padding_mask=mask)
        ha = self.a_proj(audio)
        hv = self.v_proj(visual)
        s = self.spk(spk)
        return ht + s, ha + s, hv + s, mask


## 4. Graph construction helpers

In [ ]:
def flatten_batch(ht, ha, hv, lengths):
    """Return list of per-dialogue stacked node features [3L, H] and a global node tensor.
    Node ordering per dialogue: [L text nodes, L audio nodes, L visual nodes].
    """
    feats = []
    offsets = []
    cur = 0
    for b, n in enumerate(lengths.tolist()):
        t = ht[b, :n]; a = ha[b, :n]; v = hv[b, :n]
        feats.append(torch.cat([t, a, v], dim=0))
        offsets.append(cur)
        cur += 3 * n
    return torch.cat(feats, dim=0), offsets  # [N_total, H]

def unflatten_batch(flat, lengths, offsets):
    """Inverse of flatten_batch. Returns three padded tensors aligned with input order."""
    B = len(lengths)
    L = int(lengths.max().item())
    H = flat.shape[-1]
    ht = flat.new_zeros(B, L, H); ha = flat.new_zeros(B, L, H); hv = flat.new_zeros(B, L, H)
    for b, n in enumerate(lengths.tolist()):
        o = offsets[b]
        ht[b, :n] = flat[o:o+n]
        ha[b, :n] = flat[o+n:o+2*n]
        hv[b, :n] = flat[o+2*n:o+3*n]
    return ht, ha, hv

def angular_weight(x, edge_index):
    """a_ij = 1 - arccos(cos)/pi  in [0,1] from ConxGNN. Stable."""
    src, dst = edge_index
    xs = F.normalize(x[src], dim=-1); xd = F.normalize(x[dst], dim=-1)
    cos = (xs * xd).sum(-1).clamp(-1 + 1e-6, 1 - 1e-6)
    return 1.0 - torch.arccos(cos) / math.pi

## 5. Inception Graph Module (Block B-IGM)
Multi-window context graph + implicit-edge detector + parallel k-GNN-ish branches.
We use `GraphConv` (Morris '19, the higher-order WL convolution available in PyG) for each branch.

In [ ]:
class ImplicitEdgeDetector(nn.Module):
    """HRG-SSA-style FFN scoring + masked softmax > 1/N threshold (per modality, per dialogue)."""
    def __init__(self, d):
        super().__init__()
        self.W = nn.Linear(2 * d, 1, bias=False)
        self.act = nn.LeakyReLU(0.2)
    def extra_edges(self, feats_per_modality, n):
        if n < 2: return None, None
        f = feats_per_modality
        fi = f.unsqueeze(1).expand(n, n, -1)
        fj = f.unsqueeze(0).expand(n, n, -1)
        s = self.act(self.W(torch.cat([fi, fj], dim=-1)).squeeze(-1))  # [n, n]
        causal = torch.tril(torch.ones(n, n, device=f.device, dtype=torch.bool))
        s = s.masked_fill(~causal, -1e9)
        a = F.softmax(s, dim=-1)
        keep = (a > (1.0 / n)) & causal
        idx = keep.nonzero(as_tuple=False)
        if idx.numel() == 0: return None, None
        return idx[:, 0], idx[:, 1]

def build_igm_graph(lengths, offsets, ht, ha, hv, p_window, f_window, implicit=None, device='cpu'):
    src, dst = [], []
    feats_for_implicit = (ht, ha, hv)
    for b, n in enumerate(lengths.tolist()):
        o = offsets[b]
        t0, a0, v0 = o, o + n, o + 2 * n
        idx_t = torch.arange(n, device=device) + t0
        idx_a = torch.arange(n, device=device) + a0
        idx_v = torch.arange(n, device=device) + v0
        for x, y in [(idx_t, idx_a), (idx_t, idx_v), (idx_a, idx_v)]:
            src.append(x); dst.append(y)
            src.append(y); dst.append(x)
        for idx in (idx_t, idx_a, idx_v):
            i_grid = torch.arange(n, device=device)
            for shift in range(1, p_window + 1):
                m = i_grid >= shift
                if m.any():
                    src.append(idx[m]); dst.append(idx[m] - shift)
                    src.append(idx[m] - shift); dst.append(idx[m])
            for shift in range(1, f_window + 1):
                m = (i_grid + shift) < n
                if m.any():
                    src.append(idx[m]); dst.append(idx[m] + shift)
                    src.append(idx[m] + shift); dst.append(idx[m])
        if implicit is not None:
            for mod_idx, base in enumerate([t0, a0, v0]):
                mod_feats = feats_for_implicit[mod_idx][b, :n]
                u, w = implicit.extra_edges(mod_feats, n)
                if u is not None:
                    u = u + base; w = w + base
                    src.append(u); dst.append(w)
                    src.append(w); dst.append(u)
    if len(src) == 0:
        return torch.zeros(2, 0, dtype=torch.long, device=device)
    src = torch.cat(src, dim=0); dst = torch.cat(dst, dim=0)
    return torch.stack([src, dst], dim=0)

class IGMBranch(nn.Module):
    """k-GNN branch built from PyG TransformerConv.
    - Multi-head attention learns the importance of each edge dynamically.
    - The angular-similarity edge weight is passed as a 1-D `edge_attr` so the
      attention has a relational prior (gated against learned attention scores via `beta=True`).
    - `concat=True` with `out_channels = d/heads` keeps the output at d so the
      downstream LayerNorm + residual + the rest of the model are unaffected.
    """
    def __init__(self, d, n_layers, heads=4, dropout=0.1):
        super().__init__()
        assert d % heads == 0, f'hidden {d} not divisible by heads {heads}'
        d_head = d // heads
        self.convs = nn.ModuleList([
            TransformerConv(in_channels=d, out_channels=d_head, heads=heads,
                            concat=True, beta=True, dropout=dropout, edge_dim=1)
            for _ in range(n_layers)
        ])
        self.norms = nn.ModuleList([nn.LayerNorm(d) for _ in range(n_layers)])
    def forward(self, x, edge_index, edge_weight=None):
        # TransformerConv consumes edge_attr of shape [E, edge_dim]; lift scalar→[E,1]
        edge_attr = None if edge_weight is None else edge_weight.unsqueeze(-1)
        h = x
        for conv, ln in zip(self.convs, self.norms):
            h_new = conv(h, edge_index, edge_attr)
            h = ln(F.relu(h_new) + h)
        return h

class InceptionGraphModule(nn.Module):
    def __init__(self, d, windows, n_layers, heads=4, dropout=0.1):
        super().__init__()
        self.windows = windows
        self.branches = nn.ModuleList([IGMBranch(d, n_layers, heads=heads, dropout=dropout)
                                       for _ in windows])
        self.implicit = ImplicitEdgeDetector(d)
    def forward(self, flat, lengths, offsets, ht, ha, hv):
        outs = []
        for (p, f), branch in zip(self.windows, self.branches):
            edge_index = build_igm_graph(lengths, offsets, ht, ha, hv,
                                         p, f, implicit=self.implicit, device=flat.device)
            ew = angular_weight(flat, edge_index)
            outs.append(branch(flat, edge_index, ew))
        return torch.stack(outs, dim=0).mean(dim=0)


## 6. Hypergraph Module (Block B-HM, M3Net-style)
Two kinds of hyperedges per dialogue:
- 3 modality-spanning hyperedges (one per modality, all L utterance nodes)
- L utterance-spanning hyperedges (3 modality nodes of each utterance)

Edge-dependent node weights `γ_e(v)` are realised via the per-edge `EW_weight` learnable vector (the M3Net incidence-matrix entries).

In [ ]:
class HypergraphConvM3(nn.Module):
    """M3Net-style hyperedge conv via two scatter_adds:
        edge feat:  e = sum_v  B[v,e] * (W x[v])           (node -> edge)
        node feat:  v = sum_e  D[v,e] * gamma_e(v) * e     (edge -> node)
    where B and D are normalizers from M3Net (Eq.5). Avoids PyG propagate flow tricks.
    """
    def __init__(self, d_in, d_out):
        super().__init__()
        self.lin = nn.Linear(d_in, d_out, bias=False)
        self.bias = nn.Parameter(torch.zeros(d_out))

    def forward(self, x, hyperedge_index, hyperedge_weight=None, ew_weight=None):
        N = x.size(0)
        if hyperedge_index.numel() == 0:
            return F.leaky_relu(self.lin(x) + self.bias)
        node_idx, edge_idx = hyperedge_index[0], hyperedge_index[1]
        E = int(edge_idx.max().item()) + 1
        if hyperedge_weight is None:
            hyperedge_weight = x.new_ones(E)
        # D[v] = 1 / sum_e w(e)        (per node)
        D = scatter_add(hyperedge_weight[edge_idx], node_idx, dim=0, dim_size=N)
        D = 1.0 / D; D[D == float('inf')] = 0
        # B[e] = 1 / sum_v gamma_e(v)  (per edge)
        if ew_weight is None:
            B = scatter_add(x.new_ones(hyperedge_index.size(1)), edge_idx, dim=0, dim_size=E)
        else:
            B = scatter_add(ew_weight[node_idx], edge_idx, dim=0, dim_size=E)
        B = 1.0 / B; B[B == float('inf')] = 0
        h = self.lin(x)                                   # [N, d]
        # node -> edge: messages weighted by B[e]
        msg_n2e = h[node_idx] * B[edge_idx].unsqueeze(-1) # [P, d]
        out_e   = scatter_add(msg_n2e, edge_idx, dim=0, dim_size=E)  # [E, d]
        # edge -> node: messages weighted by D[v]
        msg_e2n = out_e[edge_idx] * D[node_idx].unsqueeze(-1)        # [P, d]
        out_n   = scatter_add(msg_e2n, node_idx, dim=0, dim_size=N)  # [N, d]
        return F.leaky_relu(out_n + self.bias)

def build_hyperedge_index(lengths, offsets, device):
    nodes, edges = [], []
    edge_count = 0
    for b, n in enumerate(lengths.tolist()):
        o = offsets[b]
        t0, a0, v0 = o, o + n, o + 2 * n
        # 3 modality hyperedges
        for base in (t0, a0, v0):
            for k in range(n):
                nodes.append(base + k); edges.append(edge_count)
            edge_count += 1
        # L utterance hyperedges
        for k in range(n):
            nodes.extend([t0 + k, a0 + k, v0 + k])
            edges.extend([edge_count] * 3)
            edge_count += 1
    if len(nodes) == 0:
        return torch.zeros(2, 0, dtype=torch.long, device=device)
    return torch.tensor([nodes, edges], dtype=torch.long, device=device)

class HyperGraphModule(nn.Module):
    def __init__(self, d, n_layers, max_nodes=50_000, max_edges=50_000):
        super().__init__()
        self.convs = nn.ModuleList([HypergraphConvM3(d, d) for _ in range(n_layers)])
        # Learnable per-edge weight w(e) and per-node weight gamma_e(v) (M3Net Eq.5).
        self.hyperedge_weight = nn.Parameter(torch.ones(max_edges))
        self.ew_weight        = nn.Parameter(torch.ones(max_nodes))
    def forward(self, flat, lengths, offsets):
        hi = build_hyperedge_index(lengths, offsets, flat.device)
        E = int(hi[1].max().item()) + 1
        N = flat.size(0)
        assert E <= self.hyperedge_weight.numel(), f'too many hyperedges ({E})'
        assert N <= self.ew_weight.numel(),       f'too many nodes ({N})'
        w  = self.hyperedge_weight[:E]
        ew = self.ew_weight[:N]
        h = flat
        for conv in self.convs:
            h = conv(h, hi, w, ew)
        return h


## 7. Multi-Frequency Module (Block C, M3Net §3.3)
Low-pass `I + D^{-1/2} A D^{-1/2}` + high-pass `I - D^{-1/2} A D^{-1/2}` with edge-wise self-gating from M3Net's `highConv`.

In [ ]:
class HighFreqConv(MessagePassing):
    def __init__(self, d):
        super().__init__(aggr='add')
        self.gate = nn.Linear(2 * d, 1)
    def forward(self, x, edge_index):
        return self.propagate(edge_index, size=(x.size(0), x.size(0)), x=x)
    def message(self, x_i, x_j, edge_index, size):
        row, col = edge_index
        deg_ = degree(row, size[0], dtype=x_j.dtype)
        d_inv = deg_.pow(-0.5)
        norm  = d_inv[row] * d_inv[col]
        a_g = torch.tanh(self.gate(torch.cat([x_i, x_j], dim=1)))
        return norm.view(-1, 1) * x_j * a_g

def build_mf_edges(lengths, offsets, device):
    """Build per-batch fully-connected within-dialogue per-modality + cross-modal same-utterance."""
    src, dst = [], []
    for b, n in enumerate(lengths.tolist()):
        o = offsets[b]
        t0, a0, v0 = o, o + n, o + 2 * n
        for base in (t0, a0, v0):
            for pair in permutations(range(n), 2):
                src.append(base + pair[0]); dst.append(base + pair[1])
        # cross-modal, same utterance
        for k in range(n):
            ids = [t0 + k, a0 + k, v0 + k]
            for x, y in permutations(ids, 2):
                src.append(x); dst.append(y)
    if len(src) == 0:
        return torch.zeros(2, 0, dtype=torch.long, device=device)
    return torch.tensor([src, dst], dtype=torch.long, device=device)

class MultiFrequencyModule(nn.Module):
    def __init__(self, d, k_layers):
        super().__init__()
        self.convs = nn.ModuleList([HighFreqConv(d) for _ in range(k_layers)])
    def forward(self, flat, lengths, offsets):
        ei = build_mf_edges(lengths, offsets, flat.device)
        h = flat
        for c in self.convs:
            h = h + c(h, ei)
        return h

## 8. Cross-Modal Attention Fusion (Block E) + Classifier (Block F)

In [ ]:
class CrossModalAttn(nn.Module):
    """Project per-modality 3d -> d, run text-anchored CA at d, then concat 3*d for clf."""
    def __init__(self, d_in, d_h, heads=4, dropout=0.3):
        super().__init__()
        self.proj_t = nn.Linear(d_in, d_h)
        self.proj_a = nn.Linear(d_in, d_h)
        self.proj_v = nn.Linear(d_in, d_h)
        self.va = nn.MultiheadAttention(d_h, heads, dropout=dropout, batch_first=True)
        self.aa = nn.MultiheadAttention(d_h, heads, dropout=dropout, batch_first=True)
    def forward(self, mt, ma, mv, key_padding_mask):
        mt = F.gelu(self.proj_t(mt))
        ma = F.gelu(self.proj_a(ma))
        mv = F.gelu(self.proj_v(mv))
        v_to_t, _ = self.va(mt, mv, mv, key_padding_mask=key_padding_mask)
        a_to_t, _ = self.aa(mt, ma, ma, key_padding_mask=key_padding_mask)
        fhat_t = mt + v_to_t + a_to_t
        return torch.cat([fhat_t, ma, mv], dim=-1)

class Classifier(nn.Module):
    def __init__(self, d_in, n_classes, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, d_in // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(d_in // 2, n_classes))
    def forward(self, z): return self.net(z)


## 9. HyFIN-Net assembly

In [ ]:
class HyFINNet(nn.Module):
    def __init__(self, cfg, d_t, d_a, d_v):
        super().__init__()
        d = cfg.hidden; ds = cfg.dataset
        self.cfg = cfg
        self.encoder = UnimodalEncoder(d_t, d_a, d_v, d, cfg.n_speakers[ds], cfg.dropout[ds])
        self.igm = InceptionGraphModule(d, cfg.igm_branches[ds], cfg.igm_layers,
                                        heads=cfg.igm_heads, dropout=cfg.dropout[ds])
        self.hm  = HyperGraphModule(d, cfg.hm_layers[ds], cfg.max_hm_nodes, cfg.max_hm_edges)
        self.mf  = MultiFrequencyModule(d, cfg.mf_layers[ds])
        d_cat = 3 * d                          # per-modality concat [p|q|f]
        self.cross = CrossModalAttn(d_in=d_cat, d_h=d, heads=cfg.ca_heads, dropout=cfg.dropout[ds])
        self.clf = Classifier(3 * d, cfg.n_classes[ds], cfg.dropout[ds])

    def _features(self, batch):
        text   = batch['text'].to(device)
        audio  = batch['audio'].to(device)
        visual = batch['visual'].to(device)
        spk    = batch['speaker'].to(device)
        lens   = batch['lengths'].to(device)
        ht, ha, hv, key_pad_mask = self.encoder(text, audio, visual, spk, lens)
        flat, offsets = flatten_batch(ht, ha, hv, lens)
        p_flat = self.igm(flat, lens, offsets, ht, ha, hv)
        q_flat = self.hm(flat, lens, offsets)
        f_flat = self.mf(flat, lens, offsets)
        m_flat = torch.cat([p_flat, q_flat, f_flat], dim=-1)
        mt, ma, mv = unflatten_batch(m_flat, lens, offsets)
        fused = self.cross(mt, ma, mv, key_padding_mask=key_pad_mask)
        return fused, key_pad_mask, lens

    def forward(self, batch, return_repr=False):
        fused, key_pad_mask, lens = self._features(batch)
        # z is the per-utterance penultimate representation fed to the classifier:
        # the fused [N, 3d] vector (text-anchored CA output concat [f^_t | a | v]).
        z = fused[~key_pad_mask]               # [N, 3d]
        logits = self.clf(z)
        if return_repr:
            # dialogue id per utterance, aligned with fused[~key_pad_mask]'s
            # row-major (dialogue-grouped) order so the contrastive loss can keep
            # positive/anchor pairs inside dialogue boundaries.
            B = lens.size(0)
            dialogue_ids = torch.arange(B, device=z.device).repeat_interleave(lens)
            return logits, z, dialogue_ids
        return logits


## 10. Losses — CBCE + CBFC + DualCL

- **CBCE** — class-balanced (effective-number weighted) cross-entropy with label
  smoothing on the classifier logits (pointwise).
- **CBFC** — supervised **focal-contrastive** loss (ConxGNN Eq.18) over the
  penultimate fused `[N, 3d]` representations. For each anchor it forms
  `t_{j,k} = softmax_{k≠j} sim(z_j, z_k)/τ` over the *other utterances of the
  same dialogue*, focal-weights it by `(1−t)^γ`, and pulls same-label pairs
  together — a genuine pairwise, representation-shaping term (not focal CE).
- **DualCL** — placeholder (`dualcl_lam = 0`, not yet wired into training).


In [ ]:
def effective_class_weights(class_counts, beta=0.999):
    eff = 1.0 - np.power(beta, class_counts)
    w = (1.0 - beta) / np.maximum(eff, 1e-12)
    w = w / w.sum() * len(class_counts)
    return torch.tensor(w, dtype=torch.float32)

class CBCELoss(nn.Module):
    def __init__(self, w, label_smoothing=0.0):
        super().__init__()
        self.register_buffer('w', w); self.ls = label_smoothing
    def forward(self, logits, y):
        return F.cross_entropy(logits, y, weight=self.w.to(logits.device),
                               label_smoothing=self.ls)

class CBFCLoss(nn.Module):
    """Class-Balanced Focal *Contrastive* loss over penultimate representations
    (ConxGNN Eq.18) — supervised and restricted to within-dialogue pairs.

    This is a true pairwise, representation-shaping loss (NOT the previous
    class-balanced focal cross-entropy on the classifier logits). For each anchor
    utterance j it forms the attention-like distribution over the *other*
    utterances of the same dialogue
        t_{j,k} = softmax_{k != j, same dialogue}  sim(z_j, z_k) / tau ,
    with sim = cosine similarity of the penultimate (fused [N, 3d]) vectors.
    Positives P(j) are the other utterances in the same dialogue that share j's
    label. The focal supervised-contrastive objective pulls same-label pairs
    together:
        L_j = -(1/|P(j)|) * sum_{k in P(j)} (1 - t_{j,k})^gamma * log t_{j,k} ,
    and each anchor is scaled by its class-balanced weight w[y_j]. gamma=1 gives
    the exact (1 - t) * log t form of ConxGNN Eq.18; gamma>1 sharpens the focal
    down-weighting of easy (already-close) positives.

    Inputs:
        z            [N, D]  penultimate per-utterance vectors (requires grad)
        y            [N]     integer labels
        dialogue_ids [N]     dialogue index per utterance (positives/anchors are
                             masked to the same dialogue, matching the paper; a
                             batch-wide SupCon variant would be simpler but less
                             faithful).
    """
    def __init__(self, w, gamma=2.0, temp=0.5):
        super().__init__()
        self.register_buffer('w', w); self.gamma = gamma; self.temp = temp

    def forward(self, z, y, dialogue_ids):
        N = z.size(0)
        if N < 2:
            return z.new_zeros(())
        zn  = F.normalize(z, dim=-1)
        sim = (zn @ zn.t()) / self.temp                      # [N, N] cosine / tau
        same_dia  = dialogue_ids[:, None] == dialogue_ids[None, :]
        self_mask = torch.eye(N, dtype=torch.bool, device=z.device)
        cand = same_dia & ~self_mask                         # softmax support: same dialogue, k != j
        pos  = cand & (y[:, None] == y[None, :])             # positives: + same label
        # masked log-softmax over candidate keys; non-candidates pushed to ~0 prob
        neg_inf = torch.finfo(sim.dtype).min
        logt = F.log_softmax(sim.masked_fill(~cand, neg_inf), dim=-1)   # log t_{j,k}
        t    = logt.exp()
        term = (1.0 - t).pow(self.gamma) * logt              # (1 - t)^gamma * log t
        term = term.masked_fill(~pos, 0.0)                   # keep only positive pairs
        num_pos = pos.sum(-1)                                # |P(j)|
        valid   = num_pos > 0                                # anchors with >=1 positive
        if not valid.any():
            return z.new_zeros(())
        per_anchor = -term.sum(-1) / num_pos.clamp(min=1)    # [N]
        wj = self.w.to(z.device)[y]
        return (wj * per_anchor)[valid].sum() / valid.sum().clamp(min=1)


## 11. Train / evaluate

In [ ]:
@torch.no_grad()
def evaluate(model, loader, loss_fns=None):
    """Returns (acc, wf1, mf1, y_true, y_pred, mean_loss).
    If loss_fns=(cbce, cbfc, mu) is given, also computes the mean per-batch loss
    (same CBCE + mu*CBFC objective used in training) so we can plot a val-loss
    curve; otherwise mean_loss is None. CBFC is the supervised focal-contrastive
    loss, so it needs the penultimate reps + dialogue ids (return_repr=True).
    """
    model.eval()
    ys, ps = [], []
    total_loss, nb = 0.0, 0
    for batch in loader:
        if loss_fns is not None:
            cbce_fn, cbfc_fn, mu = loss_fns
            logits, z, dia = model(batch, return_repr=True)
            y = batch['labels'].to(device)
            l_ce = cbce_fn(logits, y)
            l_fc = cbfc_fn(z, y, dia) if mu > 0 else logits.new_zeros(())
            total_loss += (l_ce + mu * l_fc).item(); nb += 1
        else:
            logits = model(batch)
        ps.append(logits.argmax(-1).cpu().numpy())
        ys.append(batch['labels'].numpy())
    ys = np.concatenate(ys); ps = np.concatenate(ps)
    acc = accuracy_score(ys, ps)
    wf1 = f1_score(ys, ps, average='weighted')
    mf1 = f1_score(ys, ps, average='macro')
    mean_loss = (total_loss / max(1, nb)) if loss_fns is not None else None
    return acc, wf1, mf1, ys, ps, mean_loss

def make_scheduler(optim, cfg, steps_per_epoch):
    warmup_steps = cfg.warmup_epochs * steps_per_epoch
    total_steps  = cfg.epochs * steps_per_epoch
    def lr_lambda(step):
        if step < warmup_steps:
            return float(step + 1) / float(max(1, warmup_steps))
        progress = (step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return 0.5 * (1.0 + math.cos(math.pi * progress))
    return torch.optim.lr_scheduler.LambdaLR(optim, lr_lambda)

def train(cfg, raw):
    model = HyFINNet(cfg, D_T, D_A, D_V).to(device)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'#params: {n_params/1e6:.2f}M')
    w = effective_class_weights(class_counts, beta=cfg.beta_cb).to(device)
    print('CB weights:', [f'{x:.3f}' for x in w.tolist()])
    cbce = CBCELoss(w, label_smoothing=cfg.label_smoothing)
    cbfc = CBFCLoss(w, gamma=cfg.cbfc_gamma, temp=cfg.cbfc_temp)
    optim = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    sched = make_scheduler(optim, cfg, steps_per_epoch=len(train_loader))
    best_val  = {'wf1': -1, 'epoch': -1}
    best_test = {'wf1': -1, 'epoch': -1}
    history = []
    for ep in range(1, cfg.epochs + 1):
        model.train()
        t0 = time.time(); running = 0.0; nb = 0
        for step, batch in enumerate(train_loader):
            optim.zero_grad()
            logits, z, dia = model(batch, return_repr=True)
            y = batch['labels'].to(device)
            l_ce = cbce(logits, y)
            l_fc = cbfc(z, y, dia) if cfg.cbfc_mu > 0 else logits.new_zeros(())
            loss = l_ce + cfg.cbfc_mu * l_fc
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            optim.step(); sched.step()
            running += loss.item(); nb += 1
            if (step + 1) % cfg.log_every == 0:
                print(f'  ep{ep} step{step+1}/{len(train_loader)} '
                      f'ce={l_ce.item():.4f} fc={l_fc.item():.4f} lr={sched.get_last_lr()[0]:.2e}')
        tr_loss = running / max(1, nb)
        acc_v, wf1_v, mf1_v, _, _, loss_v = evaluate(
            model, val_loader, loss_fns=(cbce, cbfc, cfg.cbfc_mu))
        acc_t, wf1_t, mf1_t, *_ = evaluate(model, test_loader)
        dt = time.time() - t0
        marker_v = ''
        marker_t = ''
        if wf1_v > best_val['wf1']:
            best_val = {'wf1': wf1_v, 'epoch': ep, 'test_acc': acc_t,
                        'test_wf1': wf1_t, 'test_mf1': mf1_t}
            torch.save({'state_dict': model.state_dict(), 'cfg': cfg.__dict__,
                        'dims': (D_T, D_A, D_V), 'epoch': ep, 'best_val': best_val},
                       os.path.join(cfg.save_dir, f'hyfin_{cfg.dataset}_bestval.pt'))
            marker_v = '  [BEST-VAL]'
        if wf1_t > best_test['wf1']:
            best_test = {'wf1': wf1_t, 'epoch': ep, 'val_wf1': wf1_v,
                         'test_acc': acc_t, 'test_mf1': mf1_t}
            torch.save({'state_dict': model.state_dict(), 'cfg': cfg.__dict__,
                        'dims': (D_T, D_A, D_V), 'epoch': ep, 'best_test': best_test},
                       os.path.join(cfg.save_dir, f'hyfin_{cfg.dataset}_besttest.pt'))
            marker_t = '  [BEST-TEST]'
        print(f'[ep{ep:02d}] {dt:.1f}s  train_loss={tr_loss:.4f} val_loss={loss_v:.4f}  '
              f'val acc={acc_v:.4f} wF1={wf1_v:.4f} mF1={mf1_v:.4f}  '
              f'test acc={acc_t:.4f} wF1={wf1_t:.4f} mF1={mf1_t:.4f}{marker_v}{marker_t}')
        history.append({'epoch': ep, 'loss': tr_loss, 'val_loss': loss_v,
                        'val_acc': acc_v, 'val_wf1': wf1_v, 'val_mf1': mf1_v,
                        'test_acc': acc_t, 'test_wf1': wf1_t, 'test_mf1': mf1_t})
    with open(os.path.join(cfg.save_dir, f'hyfin_{cfg.dataset}_history.json'), 'w') as f:
        json.dump({'history': history, 'best_val': best_val, 'best_test': best_test}, f, indent=2)
    print(f'\nBEST-VAL  : ep{best_val["epoch"]:>3}  test wF1={best_val.get("test_wf1", 0):.4f}')
    print(f'BEST-TEST : ep{best_test["epoch"]:>3}  test wF1={best_test["wf1"]:.4f}')
    return model, history, best_val, best_test


## 12. Run

In [ ]:
model, history, best_val, best_test = train(cfg, raw)
print('BEST-VAL :', best_val)
print('BEST-TEST:', best_test)


In [ ]:
# --- Loss curve: train vs val ---
import matplotlib.pyplot as plt

epochs   = [h['epoch']    for h in history]
tr_loss  = [h['loss']     for h in history]
val_loss = [h.get('val_loss') for h in history]

plt.figure(figsize=(7, 5))
plt.plot(epochs, tr_loss,  marker='o', ms=3, lw=1.5, label='train loss')
plt.plot(epochs, val_loss, marker='s', ms=3, lw=1.5, label='val loss')
# mark the best-val epoch used for checkpoint selection
if best_val.get('epoch', -1) > 0:
    plt.axvline(best_val['epoch'], color='grey', ls='--', lw=1,
                label=f"best-val ep{best_val['epoch']}")
plt.xlabel('epoch'); plt.ylabel('loss (CBCE + mu*CBFC)')
plt.title(f'HyFIN-Net loss curve — {cfg.dataset}')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()

out_png = os.path.join(cfg.save_dir, f'hyfin_{cfg.dataset}_loss_curve.png')
plt.savefig(out_png, dpi=150, bbox_inches='tight')
plt.show()
print('saved loss curve ->', out_png)


In [ ]:
# Report with best-val checkpoint (the proper benchmark)
ckpt = torch.load(os.path.join(cfg.save_dir, f'hyfin_{cfg.dataset}_bestval.pt'), map_location=device)
model.load_state_dict(ckpt['state_dict'])
acc, wf1, mf1, y_true, y_pred, _ = evaluate(model, test_loader)
print(f'[BEST-VAL ckpt]  TEST acc={acc:.4f}  weighted-F1={wf1:.4f}  macro-F1={mf1:.4f}')
if cfg.dataset == 'iemocap':
    target_names = ['hap', 'sad', 'neu', 'ang', 'exc', 'fru']
else:
    target_names = ['neutral', 'surprise', 'fear', 'sadness', 'joy', 'disgust', 'anger']
print(classification_report(y_true, y_pred, target_names=target_names, digits=4))
print('Confusion matrix:')
print(confusion_matrix(y_true, y_pred))

# Also report the upper-bound best-test ckpt for comparison
ckpt2 = torch.load(os.path.join(cfg.save_dir, f'hyfin_{cfg.dataset}_besttest.pt'), map_location=device)
model.load_state_dict(ckpt2['state_dict'])
acc2, wf12, mf12, _, _, _ = evaluate(model, test_loader)
print(f'\n[BEST-TEST ckpt] TEST acc={acc2:.4f}  weighted-F1={wf12:.4f}  macro-F1={mf12:.4f}')


In [ ]:
# ── Output Export ────────────────────────────────────────────────────────────
import csv, shutil, datetime
import seaborn as sns

run_ts  = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
out_dir = os.path.join(cfg.save_dir, f'run_{cfg.dataset}_{run_ts}')
os.makedirs(out_dir, exist_ok=True)

# 1. Classification report → txt
report_str = classification_report(y_true, y_pred, target_names=target_names, digits=4)
report_path = os.path.join(out_dir, 'classification_report.txt')
with open(report_path, 'w') as f:
    f.write(f'Dataset : {cfg.dataset}\n')
    f.write(f'Run     : {run_ts}\n')
    f.write(f'Epochs  : {cfg.epochs}\n\n')
    f.write('[BEST-VAL checkpoint — test set]\n')
    f.write(f'  acc={acc:.4f}  wF1={wf1:.4f}  mF1={mf1:.4f}\n\n')
    f.write(report_str)
    f.write(f'\n[BEST-TEST checkpoint — test set]\n')
    f.write(f'  acc={acc2:.4f}  wF1={wf12:.4f}  mF1={mf12:.4f}\n')
print('report  ->', report_path)

# 2. Confusion matrix heatmap → png
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(len(target_names)*1.1 + 1, len(target_names)*1.0 + 1))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'HyFIN-Net — {cfg.dataset} confusion (best-val ckpt)')
fig.tight_layout()
cm_path = os.path.join(out_dir, 'confusion_matrix.png')
fig.savefig(cm_path, dpi=150, bbox_inches='tight')
plt.close(fig)
print('confmat ->', cm_path)

# 3. Training history → csv
hist_csv = os.path.join(out_dir, 'training_history.csv')
if history:
    with open(hist_csv, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=history[0].keys())
        w.writeheader(); w.writerows(history)
    print('history ->', hist_csv)

# 4. Per-class F1 → json (handy for quick comparisons across runs)
from sklearn.metrics import classification_report as cr_fn
cr_dict = cr_fn(y_true, y_pred, target_names=target_names, digits=4, output_dict=True)
summary = {
    'run'      : run_ts,
    'dataset'  : cfg.dataset,
    'epochs'   : cfg.epochs,
    'hidden'   : cfg.hidden,
    'best_val' : best_val,
    'best_test': best_test,
    'final_test': {
        'acc': float(acc), 'wF1': float(wf1), 'mF1': float(mf1),
        'per_class': {k: v for k, v in cr_dict.items() if k in target_names},
    },
}
summary_path = os.path.join(out_dir, 'summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)
print('summary ->', summary_path)

# 5. Copy checkpoints + loss curve into run dir
for fname in [
    f'hyfin_{cfg.dataset}_bestval.pt',
    f'hyfin_{cfg.dataset}_besttest.pt',
    f'hyfin_{cfg.dataset}_loss_curve.png',
    f'hyfin_{cfg.dataset}_history.json',
]:
    src = os.path.join(cfg.save_dir, fname)
    if os.path.exists(src):
        shutil.copy2(src, out_dir)
        print(f'copied  -> {out_dir}/{fname}')

print(f'\nAll outputs in: {out_dir}')
